In [1]:
from vseek.video_embedding.video_clip import ViClip
import torch


texts =  [" Cause you are my universe, my everything, my sunset",
      " You are everything I wanted",
      " You still give me butterflies, my butterfly",
      "and Hayha had done his 14\nyears earlier back in 1925.",
      "He did join the\nFinnish Civil Guard",
      "was required to do one\nyear of military service,",
      "That was the extent of\nhis military experience.",
      "spotted him.",
      "allowing Hayha to spot\nenemy snipers before they",
      "At the time, every\nFinnish citizen"]

embedder = ViClip(
        pretrained_model_path= "/nas/mars/model_weights/viclip/ViClip-InternVid-10M-FLT.pth"
)

text_embeddings = []
for text in texts:
    text_embedding = embedder.get_text_embedding(text)
    print(text_embedding.shape)
    text_embedding = text_embedding/text_embedding.norm(dim=-1, keepdim=True)
    text_embeddings.append(text_embedding)

query_text = "he spotted him"
query_embedding = embedder.get_text_embedding(query_text)
query_embedding = query_embedding/query_embedding.norm(dim=-1, keepdim=True)

similarities = []
for text_embedding in text_embeddings:
    similarity = torch.nn.functional.cosine_similarity(query_embedding, text_embedding)
    similarities.append(similarity)

text_embeddings_tensor = torch.stack(text_embeddings)
similarities2 = torch.matmul(text_embeddings_tensor, query_embedding.squeeze())
print(similarities2)

print(similarities)


/home/hg22723/anaconda3/envs/vseek/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/hg22723/anaconda3/envs/vseek/lib/python3.13/site-packages/timm/models/layers/__init__.py:48: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
/home/hg22723/anaconda3/envs/vseek/lib/python3.13/site-packages/timm/models/registry.py:4: FutureWarning: Importing from timm.models.registry is deprecated, please import via timm.models
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.models", FutureWarning)


torch.Size([1, 768])
torch.Size([1, 768])
torch.Size([1, 768])
torch.Size([1, 768])
torch.Size([1, 768])
torch.Size([1, 768])
torch.Size([1, 768])
torch.Size([1, 768])
torch.Size([1, 768])
torch.Size([1, 768])
tensor([[0.6725],
        [0.7657],
        [0.7014],
        [0.6995],
        [0.6470],
        [0.7367],
        [0.7778],
        [0.9153],
        [0.6992],
        [0.6881]], device='cuda:0')
[tensor([0.6725], device='cuda:0'), tensor([0.7657], device='cuda:0'), tensor([0.7014], device='cuda:0'), tensor([0.6995], device='cuda:0'), tensor([0.6470], device='cuda:0'), tensor([0.7367], device='cuda:0'), tensor([0.7778], device='cuda:0'), tensor([0.9153], device='cuda:0'), tensor([0.6992], device='cuda:0'), tensor([0.6881], device='cuda:0')]


In [ ]:
path = "/nas/mars/dataset/vseek/dataset/lvb_window_16/fvCrE5NCsts/subtitle_embeddings.pt"
subtitle_embeddings = torch.load(path)
print(subtitle_embeddings.keys())

query = "allowing Hayha to spot\nenemy snipers before they"

query_embedding = embedder.get_text_embedding(query)
query_embedding = query_embedding/query_embedding.norm(dim=-1, keepdim=True)

key_embeddings = subtitle_embeddings.values()
key_embeddings_norm = []
key_embeddings_norm2 = []
for sub, emb in subtitle_embeddings.items():
    sub_emb = embedder.get_text_embedding(sub)
    sub_emb = sub_emb.to(query_embedding.device)
    sub_emb = sub_emb/sub_emb.norm(dim=-1, keepdim=True)
    emb = emb.to(query_embedding.device)
    emb = emb/emb.norm(dim=-1, keepdim=True)
    key_embeddings_norm.append(emb)
    key_embeddings_norm2.append(sub_emb)



key_embeddings_tensor = torch.stack(list(key_embeddings_norm))
key_embeddings_tensor2 = torch.stack(list(key_embeddings_norm2))
# which one has nan
print(torch.isnan(key_embeddings_tensor2).any())
print(torch.isnan(key_embeddings_tensor).any())
print(key_embeddings_tensor2.shape)
print(key_embeddings_tensor.shape)
similairty_matrix = key_embeddings_tensor2.squeeze() @ key_embeddings_tensor.squeeze().T
print(similairty_matrix)



similarities = torch.matmul(key_embeddings_tensor, query_embedding.squeeze())

print(similarities)
max_similarity_index = torch.argmax(similarities)
max_subtitle = list(subtitle_embeddings.keys())[max_similarity_index]
print(max_subtitle)

similarities2 = torch.matmul(key_embeddings_tensor2, query_embedding.squeeze())
print(similarities2)
max_similarity_index = torch.argmax(similarities2)
max_subtitle = list(subtitle_embeddings.keys())[max_similarity_index]
print(max_subtitle)



dict_keys(['while everyone else\nwas preoccupied', 'Hayha allegedly eliminated a\nstaggering 505 enemy soldiers,', "But a sniper named Simo Hayha\ncame to Finland's defense.", 'with the war in Europe.', 'When World War II\nbroke out in 1939,', 'the Soviet Union decided\nto invade Finland', "Today, we're looking\nat the man Simo Hayha,", 'But before we get started,\nbe sure to subscribe', 'sniper in history.', 'and his unbelievable\nmarksmanship.', 'which if accurate would make\nhim the single deadliest', 'to the Weird History Channel.', 'Simo would have\nwanted it that way.', 'earned him the\nnickname Belaya Smert,', 'grew to be straight\nup terrified of Hayha.', 'Having to patrol the\nblanched Finnish wilderness', 'with the knowledge that Hayha\ncould be out there waiting', 'dipping them with a\nspectacular long distant shot', 'As you might imagine, the\ninvading Soviet soldiers', "All right, let's get sniping.", "However, Hayha's\nfellow Finnish soldiers", 'Magic Shooter.', "Isn't Ma

: 